In [1]:
import pandas as pd
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, default_converter
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr
import numpy as np
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import MNLogit
import re

_cv = default_converter + pandas2ri.converter
ro.conversion.set_conversion(_cv)

lavaan = importr('lavaan')

Error importing in API mode: ImportError('On Windows, cffi mode "ANY" is only "ABI".')
Trying to import in ABI mode.


In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.float_format", "{:.5f}".format)

# Path Analysis & Multinomial Logistic Regression

In [3]:
# ── Group definitions (by category base name) ──
restriction_groups = {'cleanliness': ['Toilets'],
                      'comfort':     ['Air Conditioning', 'Heating', 'Windows'],
                      'wifi':        ['WiFi']}
pm_groups = {'cleanliness': ['toilet'],
             'comfort':     ['climate', 'interior', 'reliability'],
             'services':    ['comms', 'wifi']}
cm_groups = {'cleanliness': ['toilet', 'cleaning'],
             'comfort':     ['climate', 'interior'],
             'wifi':        ['wifi']}

In [4]:
main_block = pd.read_parquet("../data/analysis/individual/main_block_individual.parquet")
main_block = main_block.reset_index(drop=True)
metadata_columns = ['trainset', 'train_service', 'origin_theoretical_time', 'destination_theoretical_time']
main_block = main_block.drop(columns=metadata_columns)

name_mapping = {
    'question_recommendation_score': 'Recommendation',
    'question_overall_rating': 'Satisfaction',
    'question_ticket_price_satisfa': 'Price_Sat',
    'question_overall_satisfaction_overall_service_from_eurostar_staff': 'Staff_Sat',
    'question_overall_satisfaction_journey_punctuality': 'Punctuality_Sat',
    'question_overall_satisfaction_booking_experience': 'Booking_Sat',
    'question_overall_satisfaction_information_provided_to_you_before_travelling': 'Info_Sat',
    'question_overall_satisfaction_experience_at_departure_station': 'Departure_Sat',
    'question_overall_satisfaction_comfort_onboard_the_train': 'Comfort_Sat',
    'question_overall_satisfaction_cleanliness_onboard_the_train': 'Cleanliness_Sat',
    'question_overall_satisfaction_wifi_onboard_the_train': 'WiFi_Sat'
}

main_block.rename(columns=name_mapping, inplace=True)
sat_cols = main_block.columns[main_block.columns.str.endswith('_Sat')].tolist()
main_block_questions = sat_cols + ['Satisfaction']
main_block = main_block.dropna(subset=main_block_questions)
print(f"Sample size after dropping missing values: {len(main_block)}")

Sample size after dropping missing values: 79899


In [5]:
main_block.head()

,question_qid61_nps_group,Recommendation,Staff_Sat,Punctuality_Sat,Booking_Sat,Info_Sat,Departure_Sat,Comfort_Sat,Cleanliness_Sat,Satisfaction,Price_Sat,question_strategic_pillar_ass_travelling_by_eurostar_is_an_environmentally_sustainable_option,question_strategic_pillar_ass_eurostar_manages_its_food_and_drink_offering_at_station_lounges_and_onboard_in_an_environmentally_sustainable_way,WiFi_Sat,service_id,equipment_type,route_type,restriction_open_Toilets,restriction_open_Air Conditioning,restriction_open_Refrigeration,restriction_open_Windows,restriction_open_WiFi,restriction_open_Heating,total restrictions,restriction_days_Toilets,restriction_days_Air Conditioning,restriction_days_Refrigeration,restriction_days_Windows,restriction_days_WiFi,restriction_days_Heating,longest restriction,pm_days_since_catering,pm_days_since_toilet,pm_days_since_climate,pm_days_since_interior,pm_days_since_reliability,pm_days_since_comms,pm_days_since_wifi,average days since last exams,pm_has_prior_catering,pm_has_prior_toilet,pm_has_prior_climate,pm_has_prior_interior,pm_has_prior_reliability,pm_has_prior_comms,pm_has_prior_wifi,cm_open_climate,cm_open_wifi,cm_open_interior,cm_open_catering,cm_open_toilet,cm_open_cleaning,total open faults,clean_score_routine,clean_hours_since_routine,clean_score_deep,clean_days_since_deep,last_clean_score,hours_since_last_clean,clean_has_prior_routine,clean_has_prior_deep,lda_hours_since,n_trainsets,Exceeded Rotation Time,early_journey_delay_minute,arrival_delay_minute,compensation_liability_evouchers,compensation_liability_cash
0,Detractor,2.00000,3,3,1,1,3,1,1,1,1,1,1,2,9937_20250101,TGH,Continental,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28.00000,NaN,10.00000,19.00000,0,0,0,0,1,0,1,11.00000,0.00000,38.00000,8.00000,17.00000,3.00000,77.00000,NaN,NaN,NaN,NaN,NaN,NaN,0,0,11.71667,1,00:00:00,0.00000,8.00000,0.00,0.00
1,Passive,8.00000,4,4,4,3,2,4,4,5,4,5,5,4,9937_20250101,TGH,Continental,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28.00000,NaN,10.00000,19.00000,0,0,0,0,1,0,1,11.00000,0.00000,38.00000,8.00000,17.00000,3.00000,77.00000,NaN,NaN,NaN,NaN,NaN,NaN,0,0,11.71667,1,00:00:00,0.00000,8.00000,0.00,0.00
2,Promoter,9.00000,5,5,5,5,5,5,5,4,5,5,4,5,9937_20250101,TGH,Continental,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28.00000,NaN,10.00000,19.00000,0,0,0,0,1,0,1,11.00000,0.00000,38.00000,8.00000,17.00000,3.00000,77.00000,NaN,NaN,NaN,NaN,NaN,NaN,0,0,11.71667,1,00:00:00,0.00000,8.00000,0.00,0.00
3,Detractor,5.00000,3,1,3,1,1,1,3,1,1,4,<NA>,1,9351_20250101,TGH,Continental,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27.00000,NaN,127.00000,77.00000,0,0,0,0,1,0,1,4.00000,0.00000,22.00000,9.00000,11.00000,4.00000,50.00000,NaN,NaN,NaN,NaN,NaN,NaN,0,0,14.36667,1,00:00:00,20.00000,37.00000,0.00,0.00
4,Detractor,5.00000,3,1,3,4,2,3,2,3,4,4,3,2,9351_20250101,TGH,Continental,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,27.00000,NaN,127.00000,77.00000,0,0,0,0,1,0,1,4.00000,0.00000,22.00000,9.00000,11.00000,4.00000,50.00000,NaN,NaN,NaN,NaN,NaN,NaN,0,0,14.36667,1,00:00:00,20.00000,37.00000,0.00,0.00


In [6]:
main_block.isna().sum()

question_qid61_nps_group                                                                                                                               0
Recommendation                                                                                                                                         0
Staff_Sat                                                                                                                                              0
Punctuality_Sat                                                                                                                                        0
Booking_Sat                                                                                                                                            0
Info_Sat                                                                                                                                               0
Departure_Sat                                                                     

In [ ]:
out = pd.DataFrame(index=main_block.index)

def slug(s): # 'Air Conditioning' -> 'air_conditioning'
    return re.sub(r'\W+', '_', s.strip().lower()).strip('_')

restriction_cats = list(dict.fromkeys(c for cats in restriction_groups.values() for c in cats))
pm_cats          = list(dict.fromkeys(c for cats in pm_groups.values()          for c in cats))
cm_cats          = list(dict.fromkeys(c for cats in cm_groups.values()          for c in cats))

# 1) RESTRICTIONS: per-category indicator + duration (not open -> 0)
for c in restriction_cats:
    s = slug(c)
    out[f'restr_open_{s}'] = main_block[f'restriction_open_{c}']
    out[f'restr_days_{s}'] = main_block[f'restriction_days_{c}'].fillna(0)

# 2) PREVENTIVE MAINTENANCE: per-category days since (no prior -> 2x max)
def never_as_max2(values, present):
    """time-since encoding: real durations where it happened; 'never' -> 2x max
       (extreme/overdue), distinct from a genuine 0 ('just serviced'). No indicator needed."""
    v = pd.to_numeric(values, errors='coerce')
    sent = v[present].max() * 2 if present.any() else 0.0
    return v.where(present, sent).fillna(sent).astype(float)

for c in pm_cats:
    s = slug(c)
    has = main_block[f'pm_has_prior_{c}'] == 1
    out[f'pm_days_{s}'] = never_as_max2(main_block[f'pm_days_since_{c}'], has)
    # out[f'pm_days_{s}'] = main_block[f'pm_days_since_{c}']

# 3) CORRECTIVE MAINTENANCE: per-category open faults (no missing)
for c in cm_cats:
    out[f'cm_open_{slug(c)}'] = main_block[f'cm_open_{c}']

# 3b) Operational outcomes
out['early_journey_delay_minute'] = main_block['early_journey_delay_minute']
out['arrival_delay_minute']       = main_block['arrival_delay_minute']
out['compensation_evouchers'] = main_block['compensation_liability_evouchers'].astype(float)
out['compensation_cash']      = main_block['compensation_liability_cash'].astype(float)
out['exceeded_rotation_minutes'] = main_block['Exceeded Rotation Time'].apply(lambda t: t.hour * 60 + t.minute + t.second / 60 if pd.notna(t) else 0)

# 3c) Cleaning -> presence indicator + interaction terms (missing -> 2x max)
has_deep_clean = main_block['clean_has_prior_deep'] == 1
out['last_clean_score']       = main_block['last_clean_score'].fillna(0)
out['days_since_deep_clean'] = never_as_max2(main_block['clean_days_since_deep'], has_deep_clean)

# 4) Survey questions + NPS
out[main_block_questions] = main_block[main_block_questions]
out['service_id'] = main_block['service_id']

out['channel'] = (main_block['route_type'] == 'Channel').astype(float)
out['new_fleet'] = main_block['equipment_type'].isin(['E320', 'RUB']).astype(float)  # newer stock = 1

const_cols = [c for c in out.columns if out[c].nunique(dropna=False) <= 1]
print("Dropping constant columns:", const_cols)
out.drop(columns=const_cols, inplace=True)

Dropping constant columns: []


In [8]:
print(len(out))
out.head()

79899


,restr_open_toilets,restr_days_toilets,restr_open_air_conditioning,restr_days_air_conditioning,restr_open_heating,restr_days_heating,restr_open_windows,restr_days_windows,restr_open_wifi,restr_days_wifi,pm_days_toilet,pm_days_climate,pm_days_interior,pm_days_reliability,pm_days_comms,pm_days_wifi,cm_open_toilet,cm_open_cleaning,cm_open_climate,cm_open_interior,cm_open_wifi,early_journey_delay_minute,arrival_delay_minute,compensation_evouchers,compensation_cash,exceeded_rotation_minutes,last_clean_score,days_since_deep_clean,Staff_Sat,Punctuality_Sat,Booking_Sat,Info_Sat,Departure_Sat,Comfort_Sat,Cleanliness_Sat,Price_Sat,WiFi_Sat,Satisfaction,service_id,channel,new_fleet
0,0,0.00000,0,0.00000,0,0.00000,0,0.00000,0,0.00000,8288.00000,4754.00000,1398.00000,28.00000,6730.00000,10.00000,17.00000,3.00000,11.00000,38.00000,0.00000,0.00000,8.00000,0.00000,0.00000,0.00000,0.00000,898.77735,3,3,1,1,3,1,1,1,2,1,9937_20250101,0.00000,0.00000
1,0,0.00000,0,0.00000,0,0.00000,0,0.00000,0,0.00000,8288.00000,4754.00000,1398.00000,28.00000,6730.00000,10.00000,17.00000,3.00000,11.00000,38.00000,0.00000,0.00000,8.00000,0.00000,0.00000,0.00000,0.00000,898.77735,4,4,4,3,2,4,4,4,4,5,9937_20250101,0.00000,0.00000
2,0,0.00000,0,0.00000,0,0.00000,0,0.00000,0,0.00000,8288.00000,4754.00000,1398.00000,28.00000,6730.00000,10.00000,17.00000,3.00000,11.00000,38.00000,0.00000,0.00000,8.00000,0.00000,0.00000,0.00000,0.00000,898.77735,5,5,5,5,5,5,5,5,5,4,9937_20250101,0.00000,0.00000
3,0,0.00000,0,0.00000,0,0.00000,0,0.00000,0,0.00000,8288.00000,4754.00000,1398.00000,27.00000,6730.00000,127.00000,11.00000,4.00000,4.00000,22.00000,0.00000,20.00000,37.00000,0.00000,0.00000,0.00000,0.00000,898.77735,3,1,3,1,1,1,3,1,1,1,9351_20250101,0.00000,0.00000
4,0,0.00000,0,0.00000,0,0.00000,0,0.00000,0,0.00000,8288.00000,4754.00000,1398.00000,27.00000,6730.00000,127.00000,11.00000,4.00000,4.00000,22.00000,0.00000,20.00000,37.00000,0.00000,0.00000,0.00000,0.00000,898.77735,3,1,3,4,2,3,2,4,2,3,9351_20250101,0.00000,0.00000


In [9]:
model_syntax = """
Info_Sat        ~ Booking_Sat
Departure_Sat   ~ Booking_Sat + Info_Sat
Punctuality_Sat ~ early_journey_delay_minute + arrival_delay_minute + exceeded_rotation_minutes + Info_Sat + Departure_Sat
Cleanliness_Sat ~ pm_days_climate + last_clean_score + days_since_deep_clean
Comfort_Sat     ~ arrival_delay_minute + early_journey_delay_minute + last_clean_score + Cleanliness_Sat + Punctuality_Sat
Staff_Sat       ~ Cleanliness_Sat + Departure_Sat + Comfort_Sat
WiFi_Sat        ~ pm_days_comms + pm_days_wifi + Comfort_Sat + Staff_Sat
Price_Sat       ~ early_journey_delay_minute + arrival_delay_minute + Booking_Sat + Comfort_Sat
Satisfaction    ~ Punctuality_Sat + Comfort_Sat + Staff_Sat + Price_Sat + WiFi_Sat + Booking_Sat + Info_Sat + Departure_Sat + arrival_delay_minute + pm_days_interior + Cleanliness_Sat
"""

In [10]:
fit = lavaan.sem(
    model_syntax,
    data=out,
    estimator="MLR",
    cluster="service_id",
    meanstructure=True,
    missing='ML',
    test='bootstrap'
)

In [11]:
param_df = lavaan.parameterEstimates(fit)
path_coefficients = param_df[param_df['op'] == '~'][
    ['lhs', 'rhs', 'est', 'se', 'pvalue', 'ci.lower', 'ci.upper']
]
print(path_coefficients)

                lhs                         rhs      est      se  pvalue  \
1          Info_Sat                 Booking_Sat  0.61663 0.00518 0.00000   
2     Departure_Sat                 Booking_Sat  0.22239 0.00622 0.00000   
3     Departure_Sat                    Info_Sat  0.45926 0.00580 0.00000   
4   Punctuality_Sat  early_journey_delay_minute -0.00658 0.00130 0.00000   
5   Punctuality_Sat        arrival_delay_minute -0.01671 0.00098 0.00000   
6   Punctuality_Sat   exceeded_rotation_minutes -0.00066 0.00027 0.01447   
7   Punctuality_Sat                    Info_Sat  0.35102 0.00562 0.00000   
8   Punctuality_Sat               Departure_Sat  0.25421 0.00502 0.00000   
9   Cleanliness_Sat             pm_days_climate -0.00007 0.00001 0.00000   
10  Cleanliness_Sat            last_clean_score  0.00014 0.00029 0.62488   
11  Cleanliness_Sat       days_since_deep_clean  0.00003 0.00001 0.01260   
12      Comfort_Sat        arrival_delay_minute  0.00012 0.00024 0.61734   
13      Comf

In [12]:
with localconverter(default_converter):
    fm_r = lavaan.fitMeasures(fit)               # stays a raw R named vector

fit_measures = pd.Series(list(fm_r), index=list(fm_r.names))
print(fit_measures[['chisq', 'df', 'pvalue', 'cfi', 'tli', 'rmsea', 'srmr']])

chisq    49369.75207
df          88.00000
pvalue       0.00000
cfi          0.88813
tli          0.83983
rmsea        0.08372
srmr         0.10644
dtype: float64


In [13]:
model_syntax = """
Info_Sat        ~ Booking_Sat
Departure_Sat   ~ Booking_Sat + Info_Sat
Punctuality_Sat ~ early_journey_delay_minute + arrival_delay_minute + exceeded_rotation_minutes + Info_Sat + Departure_Sat
Cleanliness_Sat ~ pm_days_climate + days_since_deep_clean
Comfort_Sat     ~ early_journey_delay_minute + last_clean_score + Cleanliness_Sat + Punctuality_Sat
Staff_Sat       ~ Cleanliness_Sat + Departure_Sat + Comfort_Sat
WiFi_Sat        ~ pm_days_comms + pm_days_wifi + Comfort_Sat + Staff_Sat
Price_Sat       ~ early_journey_delay_minute + arrival_delay_minute + Booking_Sat + Comfort_Sat
Satisfaction    ~ Punctuality_Sat + Comfort_Sat + Staff_Sat + Price_Sat + WiFi_Sat + Booking_Sat + Info_Sat + Departure_Sat + arrival_delay_minute + pm_days_interior + Cleanliness_Sat
"""

In [14]:
out.drop(columns=['last_clean_score', 'days_since_deep_clean'], inplace=True)
out['last_clean_score'] = main_block['last_clean_score']
out['days_since_deep_clean'] = main_block['clean_days_since_deep']

fit = lavaan.sem(
    model_syntax,
    data=out,
    estimator="MLR",
    cluster="service_id",
    meanstructure=True,
    missing='ML',
    test='bootstrap'
)

In [ ]:
param_df = lavaan.parameterEstimates(fit)
path_coefficients = param_df[param_df['op'] == '~'][
    ['lhs', 'rhs', 'est', 'se', 'pvalue', 'ci.lower', 'ci.upper']
]
print(path_coefficients)

                lhs                         rhs      est  se  pvalue  \
1          Info_Sat                 Booking_Sat  0.60802 NaN     NaN   
2     Departure_Sat                 Booking_Sat  0.21689 NaN     NaN   
3     Departure_Sat                    Info_Sat  0.47160 NaN     NaN   
4   Punctuality_Sat  early_journey_delay_minute -0.00922 NaN     NaN   
5   Punctuality_Sat        arrival_delay_minute -0.01521 NaN     NaN   
6   Punctuality_Sat   exceeded_rotation_minutes -0.00053 NaN     NaN   
7   Punctuality_Sat                    Info_Sat  0.32433 NaN     NaN   
8   Punctuality_Sat               Departure_Sat  0.23885 NaN     NaN   
9   Cleanliness_Sat             pm_days_climate -0.00007 NaN     NaN   
10  Cleanliness_Sat       days_since_deep_clean -0.00016 NaN     NaN   
11      Comfort_Sat  early_journey_delay_minute  0.00255 NaN     NaN   
12      Comfort_Sat            last_clean_score  0.00109 NaN     NaN   
13      Comfort_Sat             Cleanliness_Sat  0.59537 NaN    

In [16]:
with localconverter(default_converter):
    fm_r = lavaan.fitMeasures(fit)
    
fit_measures = pd.Series(list(fm_r), index=list(fm_r.names))
print(fit_measures[['chisq', 'df', 'pvalue', 'cfi', 'tli', 'rmsea', 'srmr']])

chisq    24712.66484
df          90.00000
pvalue       0.00000
cfi          0.93899
tli          0.91459
rmsea        0.07995
srmr         0.10275
dtype: float64


In [17]:
nps_dict = {'Detractor': 1, 'Promoter': 2, 'Passive': 0}
out['nps'] = main_block['question_qid61_nps_group'].map(nps_dict)

X_logit = sm.add_constant(out[['Satisfaction', 'Staff_Sat', 'Comfort_Sat', 'Price_Sat',
                               'Info_Sat', 'Departure_Sat', 'Booking_Sat', 'Cleanliness_Sat',
                               'cm_open_wifi', 'pm_days_reliability']]).astype(float)
y_logit = out['nps'].astype(int)

mnl_model = MNLogit(y_logit, X_logit)
mnl_result = mnl_model.fit(
    cov_type='cluster',
    cov_kwds={'groups': out['service_id']},
    disp=False
)
print(mnl_result.summary())

                          MNLogit Regression Results                          
Dep. Variable:                    nps   No. Observations:                79899
Model:                        MNLogit   Df Residuals:                    79877
Method:                           MLE   Df Model:                           20
Date:                Thu, 18 Jun 2026   Pseudo R-squ.:                  0.3447
Time:                        09:16:52   Log-Likelihood:                -54092.
converged:                       True   LL-Null:                       -82544.
Covariance Type:              cluster   LLR p-value:                     0.000
              nps=1       coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                   6.7245      0.080     84.431      0.000       6.568       6.881
Satisfaction           -1.0385      0.016    -66.544      0.000      -1.069      -1.008
Staff_Sat       

In [18]:
controlled_model = """
Info_Sat        ~ Booking_Sat + channel
Departure_Sat   ~ Booking_Sat + Info_Sat + channel
Punctuality_Sat ~ early_journey_delay_minute + arrival_delay_minute + exceeded_rotation_minutes + Info_Sat + Departure_Sat + channel + new_fleet
Cleanliness_Sat ~ pm_days_climate + days_since_deep_clean + channel + new_fleet
Comfort_Sat     ~ early_journey_delay_minute + last_clean_score + Cleanliness_Sat + Punctuality_Sat + channel + new_fleet
Staff_Sat       ~ Cleanliness_Sat + Departure_Sat + Comfort_Sat + channel + new_fleet
WiFi_Sat        ~ pm_days_comms + pm_days_wifi + Comfort_Sat + Staff_Sat + channel + new_fleet
Price_Sat       ~ early_journey_delay_minute + arrival_delay_minute + Booking_Sat + Comfort_Sat + channel + new_fleet
Satisfaction    ~ Punctuality_Sat + Comfort_Sat + Staff_Sat + Price_Sat + WiFi_Sat + Booking_Sat + Info_Sat + Departure_Sat + arrival_delay_minute + pm_days_interior + channel + new_fleet + Cleanliness_Sat
"""

In [19]:
controlled_fit = lavaan.sem(controlled_model, data=out, estimator="MLR", cluster="service_id", meanstructure=True, missing='ML')
param_df = lavaan.parameterEstimates(controlled_fit)
path_coefficients = param_df[param_df['op'] == '~'][
    ['lhs', 'rhs', 'est', 'se', 'pvalue', 'ci.lower', 'ci.upper']
]
print(path_coefficients)

                lhs                         rhs      est  se  pvalue  \
1          Info_Sat                 Booking_Sat  0.60761 NaN     NaN   
2          Info_Sat                     channel  0.14857 NaN     NaN   
3     Departure_Sat                 Booking_Sat  0.21725 NaN     NaN   
4     Departure_Sat                    Info_Sat  0.47238 NaN     NaN   
5     Departure_Sat                     channel -0.30308 NaN     NaN   
6   Punctuality_Sat  early_journey_delay_minute -0.00926 NaN     NaN   
7   Punctuality_Sat        arrival_delay_minute -0.01517 NaN     NaN   
8   Punctuality_Sat   exceeded_rotation_minutes -0.00054 NaN     NaN   
9   Punctuality_Sat                    Info_Sat  0.32307 NaN     NaN   
10  Punctuality_Sat               Departure_Sat  0.23960 NaN     NaN   
11  Punctuality_Sat                     channel  0.28856 NaN     NaN   
12  Punctuality_Sat                   new_fleet  0.00211 NaN     NaN   
13  Cleanliness_Sat             pm_days_climate -0.00007 NaN    

In [20]:
with localconverter(default_converter):
    fm_r = lavaan.fitMeasures(controlled_fit)
    
fit_measures = pd.Series(list(fm_r), index=list(fm_r.names))
print(fit_measures[['chisq', 'df', 'pvalue', 'cfi', 'tli', 'rmsea', 'srmr']])

chisq    24483.65707
df          92.00000
pvalue       0.00000
cfi          0.96464
tli          0.94466
rmsea        0.07870
srmr         0.09347
dtype: float64


In [ ]:
# Total effects: direct + all indirect paths through the structural model =====

def effect_matrices(coef_df, value_col):
    p = coef_df[coef_df['op'] == '~'][['lhs', 'rhs', value_col]].dropna()
    nodes = sorted(set(p['lhs']) | set(p['rhs']))
    ix = {n: k for k, n in enumerate(nodes)}
    A = np.zeros((len(nodes), len(nodes)))
    for _, r in p.iterrows():
        A[ix[r['lhs']], ix[r['rhs']]] = r[value_col]   # source rhs -> target lhs
    T = np.linalg.inv(np.eye(len(nodes)) - A) - np.eye(len(nodes))
    direct = pd.DataFrame(A, index=nodes, columns=nodes)   # [target, source]
    total  = pd.DataFrame(T, index=nodes, columns=nodes)    # [target, source]
    return direct, total

raw_pe = lavaan.parameterEstimates(controlled_fit)
std_pe = lavaan.standardizedSolution(controlled_fit).rename(columns={'est.std': 'est'})

direct_u, total_u = effect_matrices(raw_pe, 'est')   # unstandardized
direct_s, total_s = effect_matrices(std_pe, 'est')   # standardized

# Total effect of every variable on Satisfaction (direct vs indirect, standardized and non-standardized)
sat = pd.DataFrame({
    'direct_std':   direct_s.loc['Satisfaction'],
    'indirect_std': total_s.loc['Satisfaction'] - direct_s.loc['Satisfaction'],
    'total_std':    total_s.loc['Satisfaction'],
    'total_unstd':  total_u.loc['Satisfaction'],
})
sat = sat[(sat.abs() > 1e-9).any(axis=1)].sort_values('total_std', key=abs, ascending=False)
print('Total effects ON Satisfaction (sorted by |standardized total|)')
print(sat.round(4))

Total effects ON Satisfaction (sorted by |standardized total|)
                            direct_std  indirect_std  total_std  total_unstd
Departure_Sat                  0.30170       0.08970    0.39140      0.29170
Comfort_Sat                    0.18240       0.11240    0.29480      0.29270
Info_Sat                       0.09640       0.19380    0.29020      0.28430
arrival_delay_minute          -0.18790      -0.08900   -0.27680     -0.00870
Price_Sat                      0.21660       0.00000    0.21660      0.20590
Punctuality_Sat                0.14620       0.06420    0.21050      0.16120
Cleanliness_Sat               -0.00180       0.21140    0.20960      0.21630
Booking_Sat                   -0.07370       0.25980    0.18620      0.20070
Staff_Sat                      0.13310       0.01150    0.14460      0.15350
WiFi_Sat                       0.07730       0.00000    0.07730      0.05870
early_journey_delay_minute     0.00000      -0.03080   -0.03080     -0.00150
channel      

In [22]:
controlled_mnl_features = ['Satisfaction', 'Staff_Sat', 'Comfort_Sat', 'Price_Sat', 'Info_Sat',
                'Departure_Sat', 'pm_days_reliability',
                'channel', 'new_fleet', 'Cleanliness_Sat']

X_logit_controlled = sm.add_constant(out[controlled_mnl_features]).astype(float)

mnl_controlled = MNLogit(y_logit, X_logit_controlled)
mnl_controlled_result = mnl_controlled.fit(cov_type='cluster', cov_kwds={'groups': out['service_id']}, disp=False)
print(mnl_controlled_result.summary())

                          MNLogit Regression Results                          
Dep. Variable:                    nps   No. Observations:                79899
Model:                        MNLogit   Df Residuals:                    79877
Method:                           MLE   Df Model:                           20
Date:                Thu, 18 Jun 2026   Pseudo R-squ.:                  0.3449
Time:                        09:17:02   Log-Likelihood:                -54073.
converged:                       True   LL-Null:                       -82544.
Covariance Type:              cluster   LLR p-value:                     0.000
              nps=1       coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                   6.5845      0.077     85.733      0.000       6.434       6.735
Satisfaction           -1.0263      0.015    -66.434      0.000      -1.057      -0.996
Staff_Sat       

In [23]:
odds_ratios = np.exp(mnl_controlled_result.params)
print(odds_ratios)
ci_coef = mnl_controlled_result.conf_int()         # 95% CI for coefficients
ci_or = np.exp(ci_coef)                 # 95% CI for odds ratios
print(ci_or)

                            0       1
const               723.79428 0.00041
Satisfaction          0.35833 3.19506
Staff_Sat             0.85316 1.05243
Comfort_Sat           0.84233 1.09254
Price_Sat             0.62813 1.48899
Info_Sat              0.85913 1.08412
Departure_Sat         0.83763 1.15367
pm_days_reliability   0.99954 1.00039
channel               0.95849 1.13745
new_fleet             1.03572 1.09319
Cleanliness_Sat       0.94532 1.04392
                            lower     upper
nps                                        
1   const               622.64561 841.37453
    Satisfaction          0.34765   0.36935
    Staff_Sat             0.82960   0.87740
    Comfort_Sat           0.81922   0.86609
    Price_Sat             0.61178   0.64493
    Info_Sat              0.83786   0.88095
    Departure_Sat         0.81880   0.85689
    pm_days_reliability   0.99931   0.99978
    channel               0.87354   1.05169
    new_fleet             0.96517   1.11142
    Cleanliness_

In [24]:
# Sort descending   
sorted_promoter = odds_ratios.sort_values(by=1, ascending=False)
sorted_promoter.drop(columns=0, inplace=True)
print(sorted_promoter)

                          1
Satisfaction        3.19506
Price_Sat           1.48899
Departure_Sat       1.15367
channel             1.13745
new_fleet           1.09319
Comfort_Sat         1.09254
Info_Sat            1.08412
Staff_Sat           1.05243
Cleanliness_Sat     1.04392
pm_days_reliability 1.00039
const               0.00041


In [25]:
# Sort ascending
sorted_detractor = odds_ratios.sort_values(by=0, ascending=True)
sorted_detractor.drop(columns=1, inplace=True)
print(sorted_detractor)

                            0
Satisfaction          0.35833
Price_Sat             0.62813
Departure_Sat         0.83763
Comfort_Sat           0.84233
Staff_Sat             0.85316
Info_Sat              0.85913
Cleanliness_Sat       0.94532
channel               0.95849
pm_days_reliability   0.99954
new_fleet             1.03572
const               723.79428


# What-if scenarios simulations & analysis

In [26]:
rename_dict = {'Exceeded Rotation Time': 'exceeded_rotation_minutes', 'clean_days_since_deep': 'days_since_deep_clean',
               'pm_days_since_reliability': 'pm_days_reliability', 'pm_days_since_climate': 'pm_days_climate',
               'pm_days_since_comms': 'pm_days_comms', 'pm_days_since_interior': 'pm_days_interior', 'pm_days_since_wifi': 'pm_days_wifi'}
main_block.rename(columns=rename_dict, inplace=True)
main_block['exceeded_rotation_minutes'] = main_block['exceeded_rotation_minutes'].apply(lambda t: t.hour * 60 + t.minute + t.second / 60 if pd.notna(t) else 0)
data = main_block.copy()

# Extract coefficients from the fitted models

# path analysis
pe = lavaan.parameterEstimates(controlled_fit)
reg = pe[pe['op'] == '~'] # regression paths
sem_int = pe[pe['op'] == '~1'].set_index('lhs')['est'].to_dict()
sem_coef = {}
for _, r in reg.iterrows():
    sem_coef.setdefault(r['lhs'], {})[r['rhs']] = float(r['est'])
ENDOG = list(sem_coef)                        # all endogenous variables

# multinomial logit
B = mnl_controlled_result.params
mnl_feat = [c for c in B.index if c != 'const']

nps_dict = {1: 'Detractor', 2: 'Passive', 3: 'Promoter'}
ymap = mnl_controlled_result.model._ynames_map
base_role = nps_dict[2]   # 'Passive' is reference

beta_of_role = {}
beta_of_role['Detractor'] = B.iloc[:, 0].to_dict()   # column 0 -> nps=1
beta_of_role['Promoter']  = B.iloc[:, 1].to_dict()   # column 1 -> nps=2

In [ ]:
# Build baseline dictionary: means of all variables in the model
all_vars = set(mnl_feat) | set().union(*[sem_coef[y].keys() for y in ENDOG]) | set(ENDOG)
all_vars -= {'channel', 'new_fleet'}

baseline = {}
for v in all_vars:
    if v in data.columns:
        baseline[v] = float(data[v].mean())
    else:
        baseline[v] = 0.0   # default for dummies / missing

baseline['channel']   = 0.0
baseline['new_fleet'] = 0.0

ALL_LEVERS = [
    'arrival_delay_minute',
    'early_journey_delay_minute',
    'exceeded_rotation_minutes',
    'pm_days_reliability',
    'pm_days_interior',
    'pm_days_climate',
    'days_since_deep_clean',
    'last_clean_score',
    'pm_days_comms',
    'pm_days_wifi'
]


# Define plausible ranges for each lever: 5th and 95th percentiles from the data

bounds = {}
for L in ALL_LEVERS:
    lo_data = data[L].quantile(0.05)
    hi_data = data[L].quantile(0.95)
    
    # Force all bounds to be non‑negative
    lo_data = max(0.0, lo_data)
    
    if 'delay' in L.lower() or 'exceeded' in L.lower():
        lo = max(5.0, lo_data)
        hi = max(hi_data, lo + 1)
    elif 'pm_days' in L or 'days_since' in L or 'clean_score' in L:
        lo = max(1.0, lo_data)
        hi = max(hi_data, lo + 1)
    else:
        lo = lo_data
        hi = hi_data
    
    bounds[L] = (lo, hi)

In [33]:
# NPS prediction function
def eval_nps(x: dict) -> float:

    v = baseline.copy()
    v.update(x)
    for var in all_vars:
        if var not in v:
            v[var] = baseline.get(var, 0.0)

    # calculating for satisfaction scores using path analysis coefficients
    for _ in range(len(ENDOG) + 2):
        for y in ENDOG:
            y_pred = sem_int.get(y, 0.0)
            for pred, coef in sem_coef.get(y, {}).items():
                y_pred += coef * v[pred]
            v[y] = y_pred

    # nps probabilities
    e = {}
    for role in ['Detractor', 'Passive', 'Promoter']:
        if role == base_role:
            e[role] = 1.0
        else:
            b = beta_of_role[role]
            eta = b.get('const', 0.0)
            for f in mnl_feat:
                if f in v:
                    eta += b[f] * v[f]
            e[role] = np.exp(eta)

    total = e['Detractor'] + e['Passive'] + e['Promoter']
    nps = 100 * (e['Promoter'] - e['Detractor']) / total
    return nps

In [ ]:
# Random sampling analysis for each fleet scenario
scenarios = [
    {"name": "E320 (Channel, new)",   "channel": 1, "new_fleet": 1},
    {"name": "E300 (Channel, old)",   "channel": 1, "new_fleet": 0},
    {"name": "RUB (Continental, new)","channel": 0, "new_fleet": 1},
    {"name": "TGH (Continental, old)","channel": 0, "new_fleet": 0},
]

target = 38.0
N_SAMPLES = 100000   # number of random combinations per scenario

print("Monte Carlo what‑if analysis: randomly sampling all operational levers")
print(f"Target NPS ≥ {target}")

for sc in scenarios:
    # Prepare baseline for this scenario
    base_pt = baseline.copy()
    base_pt['channel'] = sc['channel']
    base_pt['new_fleet'] = sc['new_fleet']

    # Generate random samples within bounds
    random_points = pd.DataFrame()
    for L in ALL_LEVERS:
        lo, hi = bounds[L]
        # For integer-like levers (pm_days_*, days_since_*), apply rounding:
        if 'pm_days' in L or 'days_since' in L or 'last_clean_score' in L:
            random_points[L] = np.random.randint(int(lo), int(hi)+1, N_SAMPLES).astype(float)
        else:
            random_points[L] = np.random.uniform(lo, hi, N_SAMPLES)

    # Compute NPS for each random combination
    nps_vals = []
    for _, row in random_points.iterrows():
        test = base_pt.copy()
        test.update(row.to_dict())
        nps_vals.append(eval_nps(test))
    random_points['NPS'] = nps_vals

    feasible = random_points[random_points['NPS'] >= target]
    feasible = feasible.sort_values(by='NPS', ascending=False)
    n_feas = len(feasible)

    print(f"\n{sc['name']}:")
    print(f"  Baseline NPS: {eval_nps(base_pt):.2f}")
    print(f"  Sampled combinations: {N_SAMPLES}")
    print(f"  Feasible (NPS ≥ {target}): {n_feas} ({100 * n_feas / N_SAMPLES:.1f}%)")

    if n_feas == 0:
        # Show max NPS found
        max_nps = random_points['NPS'].max()
        best = random_points.loc[random_points['NPS'].idxmax()]
        print(f"  Max NPS found in random search: {max_nps:.2f}")
        print(f"  Best lever combination:")
        for L in ALL_LEVERS:
            print(f"    {L:30s}: {best[L]:.2f} (baseline: {baseline[L]:.2f})")
    else:
        # Summary statistics of feasible lever values
        print("  Feasible lever statistics (mean ± std):")
        for L in ALL_LEVERS:
            f_mean = feasible[L].mean()
            f_std  = feasible[L].std()
            b_val  = baseline[L]
            print(f"    {L:30s}: {f_mean:8.2f} ± {f_std:8.2f}   (baseline: {b_val:.2f})")

        # Show example feasible combinations
        print(f"  Example feasible combination with the highest NPS ({feasible['NPS'].iloc[0]:.0f}):")
        example = feasible.iloc[0]
        for L in ALL_LEVERS:
            print(f"    {L:30s}: {example[L]:.2f}")

Monte Carlo what‑if analysis: randomly sampling all operational levers
Target NPS ≥ 38.0

E320 (Channel, new):
  Baseline NPS: 36.44
  Sampled combinations: 100000
  Feasible (NPS ≥ 38.0): 3609 (3.6%)
  Feasible lever statistics (mean ± std):
    arrival_delay_minute          :     8.73 ±     2.95   (baseline: 13.76)
    early_journey_delay_minute    :    17.28 ±     9.37   (baseline: 6.69)
    exceeded_rotation_minutes     :    20.57 ±     9.08   (baseline: 6.44)
    pm_days_reliability           :   332.36 ±    77.14   (baseline: 170.13)
    pm_days_interior              :   218.01 ±   124.87   (baseline: 262.55)
    pm_days_climate               :   849.21 ±   578.58   (baseline: 356.01)
    days_since_deep_clean         :    88.51 ±    50.22   (baseline: 77.69)
    last_clean_score              :    92.25 ±     4.25   (baseline: 93.39)
    pm_days_comms                 :  2568.78 ±   363.92   (baseline: 2741.90)
    pm_days_wifi                  :    85.53 ±    43.30   (baseline: 7

In [57]:
# Target: overall Satisfaction >= 4.0
target_sat = 4.0
sat_variable = 'Satisfaction'   # can be changed to other perception dimensions

print(f"Monte Carlo what‑if analysis: targeting {sat_variable} ≥ {target_sat}")

for sc in scenarios:
    base_pt = baseline.copy()
    base_pt['channel']   = sc['channel']
    base_pt['new_fleet'] = sc['new_fleet']
    
    # Generate the same kind of random points (can re‑use the ones from earlier)
    random_points = pd.DataFrame()
    for L in ALL_LEVERS:
        lo, hi = bounds[L]
        if 'pm_days' in L or 'days_since' in L or 'clean_score' in L:
            random_points[L] = np.random.randint(int(lo), int(hi)+1, N_SAMPLES).astype(float)
        else:
            random_points[L] = np.random.uniform(lo, hi, N_SAMPLES)

    sat_vals = []
    nps_vals = []
    for _, row in random_points.iterrows():
        test = base_pt.copy()
        test.update(row.to_dict())

        # Compute the satisfaction scores
        v = baseline.copy()
        v.update(test)
        for var in all_vars:
            if var not in v:
                v[var] = baseline.get(var, 0.0)
        for _ in range(len(ENDOG) + 2):
            for y in ENDOG:
                y_pred = sem_int.get(y, 0.0)
                for pred, coef in sem_coef.get(y, {}).items():
                    y_pred += coef * v[pred]
                v[y] = y_pred
        sat_vals.append(v[sat_variable])
    
    random_points[sat_variable] = sat_vals
    feasible_sat = random_points[random_points[sat_variable] >= target_sat]
    feasible_sat = feasible_sat.sort_values(by=sat_variable, ascending=False)
    
    n_feas = len(feasible_sat)
    print(f"\n{sc['name']}:")

    # Compute baseline satisfaction directly
    base_v = baseline.copy()
    base_v['channel'] = sc['channel']; base_v['new_fleet'] = sc['new_fleet']
    for _ in range(len(ENDOG)+2):
        for y in ENDOG:
            y_pred = sem_int.get(y,0.0) + sum(sem_coef.get(y,{}).get(p,0)*base_v.get(p,0) for p in sem_coef.get(y,{}))
            base_v[y] = y_pred
    
    base_sat = base_v[sat_variable]
    print(f"  Baseline {sat_variable}: {base_sat:.2f}")
    print(f"  Sampled combinations: {N_SAMPLES}")
    print(f"  Feasible ( {sat_variable} ≥ {target_sat} ): {n_feas} ({100*n_feas/N_SAMPLES:.1f}%)")
    
    if n_feas > 0:
        print(f"  Feasible lever statistics (mean ± std):")
        for L in ALL_LEVERS:
            f_mean = feasible_sat[L].mean()
            f_std  = feasible_sat[L].std()
            b_val  = baseline[L]
            print(f"    {L:30s}: {f_mean:8.2f} ± {f_std:8.2f}   (baseline: {b_val:.2f})")
        print(f"  Example feasible combination with the highest {sat_variable} score ({feasible_sat[sat_variable].iloc[0]:.2f}):")
        example = feasible_sat.iloc[0]
        for L in ALL_LEVERS:
            print(f"    {L:30s}: {example[L]:.2f}")
    else:
        max_sat = random_points[sat_variable].max()
        best = random_points.loc[random_points[sat_variable].idxmax()]
        print(f"  Max {sat_variable} found: {max_sat:.2f}")
        print(f"  Best lever combination:")
        for L in ALL_LEVERS:
            print(f"    {L:30s}: {best[L]:.2f} (baseline: {baseline[L]:.2f})")

Monte Carlo what‑if analysis: targeting Satisfaction ≥ 4.0

E320 (Channel, new):
  Baseline Satisfaction: 3.83
  Sampled combinations: 100000
  Feasible ( Satisfaction ≥ 4.0 ): 0 (0.0%)
  Max Satisfaction found: 3.94
  Best lever combination:
    arrival_delay_minute          : 5.39 (baseline: 13.76)
    early_journey_delay_minute    : 10.84 (baseline: 6.69)
    exceeded_rotation_minutes     : 6.36 (baseline: 6.44)
    pm_days_reliability           : 148.00 (baseline: 170.13)
    pm_days_interior              : 81.00 (baseline: 262.55)
    pm_days_climate               : 73.00 (baseline: 356.01)
    days_since_deep_clean         : 16.00 (baseline: 77.69)
    last_clean_score              : 88.00 (baseline: 93.39)
    pm_days_comms                 : 3063.00 (baseline: 2741.90)
    pm_days_wifi                  : 144.00 (baseline: 76.46)

E300 (Channel, old):
  Baseline Satisfaction: 3.84
  Sampled combinations: 100000
  Feasible ( Satisfaction ≥ 4.0 ): 0 (0.0%)
  Max Satisfaction found: